# 06 - Follow-up FU2: deploy teacher relabelling as the repair at 100% Syn poisoning
Pre-registration: `experiments/prereg_followup_fu2.md` (committed before any FU2 run). Code: `s10_relabel.py`.

**Provenance (read first).** In the sweep (notebook 04) at f = 1.0 every stored Syn label was BENIGN. The registered
soft repair failed 6 of 6 times, so the model stayed at A0. "threeway+relabel" was only a *forced diagnostic*
there: registered as an open question (H5, no direction predicted), thresholds 0.9 / 0.1 chosen after seeing
`main`, never a decision, never deployed, never tested. It would have passed the gate 6 of 6 times. FU2 deploys
it; that choice was made **after** seeing the diagnostic. E1 here replays the diagnostic exactly, so it is
not new evidence. New evidence: E2 after the deployed relabel at E1, and the held-out test (scored once, later).

What runs, per seed 42 / 43 / 44: one new arm `poisoned_relabel` with s3's own code, the sweep run's frozen
teacher T (read only), its A0, the same poisoned buffer and seeds. Only R1's method changes (soft ->
threeway+relabel). The clean / poisoned / soft-repair arms are the controls, read from the sweep runs, never rerun.

**Inputs (Add Input):** notebook 04's output, the same one notebook 05 used. It holds `nids_out/` with
`clean_*.parquet`, `cii/` and `runs/cii/sweep_f1.0_s{42,43,44}`. **Internet ON** (git clone).
Accelerator: **GPU T4** (required: E1 is checked weight-for-weight against the sweep's T4 run).

Run interactively, not "Save & Run All". Cells 7-8 do nothing unless `RESTORE` / `DECISIONS_COMMIT` are set.

In [ ]:
# 1. Locate code and the notebook-04 outputs (read only)
import glob, json, os, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

CLONE = Path("/kaggle/working/code")
if not CLONE.exists() and os.environ.get("NIDS_CLONE", "1") == "1":
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DevChi1l/cicddos2019-continual-nids",
                    str(CLONE)], check=False)
# marker s10_relabel.py: an older code copy (e.g. inside the notebook-04 output) is never picked
CODE_DIR = os.environ.get("NIDS_CODE") or next(
    (str(Path(p).parent) for p in [str(CLONE / "s10_relabel.py")] +
     sorted(glob.glob("/kaggle/input/**/s10_relabel.py", recursive=True)) if Path(p).exists()), None)
assert CODE_DIR, "code with s10_relabel.py not found: turn Internet ON"
sys.path.insert(0, CODE_DIR)
import s5_explain as X
git = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code:", CODE_DIR, "| git commit:", git)

SRC = X.find_dir("runs/cii/sweep_f1.0_s44/run_manifest.json", "NIDS_FU2_SRC")
assert SRC and (SRC / "clean_train.parquet").exists() and all(
    (SRC / f"runs/cii/sweep_f1.0_s{s}/T.pt").exists() for s in (42, 43, 44)), \
    "attach notebook 04's output: nids_out/ with clean_*.parquet, cii/ and runs/cii/sweep_f1.0_s42..44"
WORK = Path(os.environ.get("NIDS_WORK", "/kaggle/working/nids_out"))
FU2 = WORK / "fu2"
print("source (read only):", SRC, "\nwrites:", FU2)

def run(*args):
    cmd = [sys.executable, str(Path(CODE_DIR) / "s10_relabel.py"), *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    subprocess.run(cmd, check=True, cwd=CODE_DIR)

COMMON = ["--src", SRC, "--work", WORK, "--seeds", "42,43,44"]

def zip_fu2(name, test=False):
    # JSON/CSV under fu2/ (no buffers, no checkpoints) -> /kaggle/working/<name>; needs only this cell
    dest = WORK.parent / name
    pick = sorted(p for p in FU2.rglob("*") if p.is_file() and p.suffix in (".json", ".csv")
                  and not p.name.startswith("buffer_") and (test or p.name != "final_test.json"))
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in pick:
            zf.write(p, Path("fu2") / p.relative_to(FU2))
    print(f"{len(pick)} files -> {dest} ({dest.stat().st_size / 1e6:.2f} MB). Download it from the Output panel.")

In [ ]:
# 2. GPU, then a dry run: input hashes, frozen teacher vs run_manifest, rebuilt poisoned buffer vs the saved one
import torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"
print(torch.cuda.get_device_name(0), torch.__version__)
run("run", *COMMON, "--dry-run")

## Validation run (no test)
Per seed and update: C1 (plain stored labels); if the gate rejects it, one R1 with threeway+relabel from the last
accepted model. Accepted -> the buffer gains the update's rows; rejected -> previous model kept. A finished seed
is skipped if the cell is re-run; an unfinished one restarts (deterministic). About 1-2 min per seed on a T4.

In [ ]:
# 3. FU2 validation run, seeds 42-44
t0 = time.time()
run("run", *COMMON)
print(f"run: {time.time() - t0:.0f} s")

In [ ]:
# 4. Validation outcome per seed (gate decisions, relabel counts, E1 replay checks)
rows = []
for s in (42, 43, 44):
    d = FU2 / "cii" / f"fu2_relabel_f1.0_s{s}"
    summ = json.loads((d / "summary.json").read_text())
    print(f"s{s}: {summ['update_status']} | classes {summ['final_classes']} | P0 {summ['p0_checks']}")
    for r in json.loads((d / "poisoned_relabel" / "rounds.json").read_text()):
        tw = r["teacher_weights"] or {}
        rows.append({"seed": s, "E": r["experience"], "attempt": r["attempt"], "accepted": r["accepted"],
                     **{f"val_{k}": v for k, v in r["val"]["recall"].items()},
                     "benign_far": r["val"]["benign_far"], "relabelled": tw.get("n_relabelled"),
                     "relabelled_to_true": tw.get("n_relabelled_to_true"),
                     "reasons": "; ".join(r["gate_reasons"]) or "-"})
print(pd.DataFrame(rows).round(4).to_string(index=False))

## STOP - download BOTH zips from the Output panel (cell 6 writes them)
- `fu2_val.zip`: validation JSON/CSV only (no checkpoints, no buffers, no test). Send it; it is committed to git
  under `experiments/results/06_fu2/`.
- `fu2_restore.zip`: the three deployed models + summaries + rounds + a sha256 manifest (no test). Keep it; it lets
  test be scored in a NEW session if this one closes.

Test is scored once, only after the commit. In this session: set `DECISIONS_COMMIT` in cell 8 and run cells 8-9.
If the session closed: new session, run cell 1, then the restore cell (7), then cells 8-9. Never cells 2-6 there.

In [ ]:
# 6. Validation outputs: fu2_val.zip (JSON/CSV only) and fu2_restore.zip (models + manifest); neither holds test
assert not list(FU2.rglob("final_test.json")), "test was already scored in this folder"
zip_fu2("fu2_val.zip")
run("bundle", "--work", WORK, "--bundle", WORK.parent / "fu2_restore.zip")

In [ ]:
# 7. RESTORE (only in a NEW session after the old one closed; skip otherwise).
#    New session: Internet ON; Add Input = notebook 04's output (as before) + a private dataset made from
#    fu2_restore.zip. Run cell 1, then this cell with RESTORE = True, then cells 8-9. Never run cells 2-6 here.
RESTORE = False
if RESTORE:
    # an FU2 bundle only (an attached FU1 bundle has followup/, not fu2/, next to its manifest)
    man = [m for m in sorted(glob.glob("/kaggle/input/**/RESTORE_MANIFEST.json", recursive=True))
           if (Path(m).parent / "fu2").is_dir()] or sorted(glob.glob("/kaggle/input/**/fu2_restore.zip", recursive=True))
    assert man, "attach the dataset made from fu2_restore.zip"
    b = Path(man[0])
    run("restore", "--work", WORK, "--bundle", b.parent if b.name == "RESTORE_MANIFEST.json" else b)
else:
    print("restore skipped (RESTORE = False)")

In [ ]:
# 8. Test, ONCE. Disabled until DECISIONS_COMMIT is the sha of the commit that holds fu2_val.zip's files.
#    score-test refuses unless summary.json and rounds.json here are byte-identical to that commit's copies
#    under experiments/results/06_fu2/ and each deployed model has the hash recorded in summary.json.
DECISIONS_COMMIT = ""
if DECISIONS_COMMIT:
    subprocess.run(["git", "-C", CODE_DIR, "fetch", "--depth", "50", "origin", "main"], check=True)
    run("score-test", *COMMON, "--decisions-commit", DECISIONS_COMMIT, "--repo", CODE_DIR)
else:
    print("skipped: commit the validation files first, then set DECISIONS_COMMIT")

In [ ]:
# 9. Test files (after cell 8 ran; works in the original or a restore session - needs only cell 1)
done = sorted(FU2.rglob("final_test.json"))
if not done:
    print("no test scored yet")
else:
    assert len(done) == 3, f"test scored for {len(done)} of 3 seeds: run cell 8"
    zip_fu2("fu2_test.zip", test=True)